In [26]:
%run geo_zones

StatementMeta(, 46969f7f-d639-49aa-a349-640fb5c69e39, 20, Finished, Available, Finished, True)

In [27]:
%run manage_files

StatementMeta(, 46969f7f-d639-49aa-a349-640fb5c69e39, 26, Finished, Available, Finished, True)

In [28]:
%run manage_tables

StatementMeta(, 46969f7f-d639-49aa-a349-640fb5c69e39, 37, Finished, Available, Finished, True)

In [29]:
%run transformation

StatementMeta(, 46969f7f-d639-49aa-a349-640fb5c69e39, 52, Finished, Available, Finished, True)

In [30]:
from pyspark.sql.functions import *

StatementMeta(, 46969f7f-d639-49aa-a349-640fb5c69e39, 53, Finished, Available, Finished, False)

In [31]:
YELLOW_TRIPDATA_FOLDER_PATH = "Files/NYC_Taxi/yellow_tripdata"
YELLOW_TRIPDATA_SCHEMA = {
    "pickup_datetime": {"type": "timestamp", "nullable": False},
    "dropoff_datetime": {"type": "timestamp", "nullable": False},
    "passenger_count": {"type": "integer", "nullable": True},
    "trip_distance": {"type": "double", "nullable": True},
    "pickup_location_id": {"type": "integer", "nullable": True},
    "dropoff_location_id": {"type": "integer", "nullable": True},
    "pickup_longitude": {"type": "double", "nullable": True},
    "pickup_latitude": {"type": "double", "nullable": True},
    "dropoff_longitude": {"type": "double", "nullable": True},
    "dropoff_latitude": {"type": "double", "nullable": True},
    "fare_amount": {"type": "double", "nullable": True},
    "total_amount": {"type": "double", "nullable": True},
    "tip_amount": {"type": "double", "nullable": True}
}
YELLOW_TRIPDATA_RESULT_COLUMNS = [
    "pickup_datetime",
    "dropoff_datetime",
    "passenger_count",
    "trip_distance",
    "pickup_location_id",
    "dropoff_location_id",
    "fare_amount",
    "total_amount",
    "tip_amount"
]
YELLOW_TRIPDATA_COLUMN_MAPPING = {
    "Trip_Pickup_DateTime": "pickup_datetime",
    "tpep_pickup_datetime": "pickup_datetime",
    "Trip_Dropoff_DateTime": "dropoff_datetime",
    "tpep_dropoff_datetime": "dropoff_datetime",
    "Passenger_Count": "passenger_count",
    "Trip_Distance": "trip_distance",
    "PULocationID": "pickup_location_id",
    "DOLocationID": "dropoff_location_id",
    "Start_Lon": "pickup_longitude",
    "Start_Lat": "pickup_latitude",
    "End_Lon": "dropoff_longitude",
    "End_Lat": "dropoff_latitude",
    "Fare_Amt": "fare_amount",
    "Total_Amt": "total_amount",
    "Tip_Amt": "tip_amount"
}

TAXI_VALUE_RANGES = {
    "trip_distance": (0, None),
    "fare_amount": (0, None),
    "total_amount": (0, None),
    "tip_amount": (0, None)
}

StatementMeta(, 46969f7f-d639-49aa-a349-640fb5c69e39, 54, Finished, Available, Finished, False)

In [32]:
def get_existing_months(df, date_column):
    return [row["yyyy-mm"] for row in df.select(date_format(date_column, "yyyy-MM").alias("yyyy-mm")).distinct().collect()]

def get_loaded_months(table_name, date_column):
    if not is_table_exist(table_name, "Silver_lakehouse"):
        return []
    df = read_table(table_name, "Silver_lakehouse")
    return get_existing_months(df, date_column)

def get_new_files(file_paths, existing_months):
    return [path for path in file_paths if path[-15:-8] not in existing_months]

def get_new_parquet_files(folder_path, table_name, date_column):
    file_paths = get_parquet_files(folder_path)
    existing_months = get_loaded_months(table_name, date_column)
    return get_new_files(file_paths, existing_months)

StatementMeta(, 46969f7f-d639-49aa-a349-640fb5c69e39, 55, Finished, Available, Finished, False)

In [33]:
def is_valid_taxi_location(location_id, longitude, latitude):
    return (col(location_id).isNotNull() | ((col(longitude) != 0) & (col(latitude) != 0)))

def filter_taxi_coordinates(df):
    is_pickup_valid = is_valid_taxi_location("pickup_location_id", "pickup_longitude", "pickup_latitude")
    is_dropoff_valid = is_valid_taxi_location("dropoff_location_id", "dropoff_longitude", "dropoff_latitude")
    return df.filter(is_pickup_valid | is_dropoff_valid)

def filter_year_month(df, file_path, date_column):
    month = file_path[-15:-8]
    return df.filter(date_format(date_column, "yyyy-MM") == month)

def fill_empty_numeric(df, schema, excluded_columns=None):
    excluded_columns = excluded_columns or []
    numeric_columns = [(column, data_type) for column, data_type in schema.items() if data_type in ["integer", "double"] and column not in excluded_columns]
    for column, data_type in numeric_columns:
        if data_type == "double":
            df = df.withColumn(column, nanvl(col(column), lit(0.0)))
    return df.fillna(0, subset=[column for column, _ in numeric_columns])

def normalize_taxi(df, column_mapping, schema):
    column_types = {column: prop["type"] for column, prop in schema.items()}
    df = rename_columns(df, column_mapping)
    df = add_missing_columns(df, column_types)
    df = cast_columns(df, column_types)
    df = fill_empty_numeric(df, column_types, ["pickup_location_id", "dropoff_location_id"])
    df = select_columns(df, column_types)
    return df

def filter_taxi(df, schema, file_path):
    df = drop_null_rows(df, [column for column, prop in schema.items() if not prop["nullable"]])
    df = filter_taxi_coordinates(df)
    df = filter_column_ranges(df, TAXI_VALUE_RANGES)
    df = filter_year_month(df, file_path, "pickup_datetime")
    return df

def load_and_process_taxi_table(file_path, column_mapping, schema):
    df = read_parquet_file(file_path)
    df = normalize_taxi(df, column_mapping, schema)
    df = filter_taxi(df, schema, file_path)
    return df

StatementMeta(, 46969f7f-d639-49aa-a349-640fb5c69e39, 56, Finished, Available, Finished, False)

In [53]:
def get_special_location_ids(df):
    special_zones = {}
    for borough in ["UNKNOWN", "OUTSIDE OF NYC"]:
        zone = df.filter(upper(trim(col("Borough"))) == borough).select("LocationID").first()
        if zone is None:
            raise ValueError(f"Could not find {borough} zone")
        special_zones[borough] = zone["LocationID"]
    return special_zones["UNKNOWN"], special_zones["OUTSIDE OF NYC"]

def get_valid_location_ids(df):
    return [row["LocationID"] for row in df.select("LocationID").collect()]

StatementMeta(, 46969f7f-d639-49aa-a349-640fb5c69e39, 76, Finished, Available, Finished, False)

In [55]:
def assign_special_location_id(df, location_id, longitude, latitude, unknown_location_id, outside_nyc_location_id):
    return df.withColumn(location_id,
        when(
            col(location_id).isNull() &
            ((col(longitude) == 0.0) | (col(latitude) == 0.0)),
            unknown_location_id
        ).when(
            col(location_id).isNull() &
            (col(longitude) != 0.0) & (col(latitude) != 0.0),
            outside_nyc_location_id
        ).otherwise(col(location_id))
    )

def assign_special_location_ids(df, unknown_location_id, outside_nyc_location_id):
    for coor_type in ["pickup", "dropoff"]:
        df = assign_special_location_id(df, f"{coor_type}_location_id", f"{coor_type}_longitude", f"{coor_type}_latitude", unknown_location_id, outside_nyc_location_id)
    return df

def filter_valid_location_ids(df, valid_location_ids):
    return df.filter(col("pickup_location_id").isin(valid_location_ids) & col("dropoff_location_id").isin(valid_location_ids))

def filter_nyc_trips(df, unknown_location_id, outside_nyc_location_id):
    return df.filter(
        (~col("pickup_location_id").isin(unknown_location_id, outside_nyc_location_id)) |
        (~col("dropoff_location_id").isin(unknown_location_id, outside_nyc_location_id))
    )

def finalize_taxi_locations(df, result_columns, unknown_location_id, outside_nyc_location_id, valid_location_ids):
    df = assign_special_location_ids(df, unknown_location_id, outside_nyc_location_id)
    df = filter_valid_location_ids(df, valid_location_ids)
    df = filter_nyc_trips(df, unknown_location_id, outside_nyc_location_id)
    df = select_columns(df, result_columns)
    return df

StatementMeta(, 46969f7f-d639-49aa-a349-640fb5c69e39, 78, Finished, Available, Finished, False)

In [54]:
def process(folder_path):
    table_name = folder_path.rstrip("/").split("/")[-1]
    new_files = get_new_parquet_files(folder_path, table_name, "pickup_datetime")
    if not new_files:
        return
    zones_df = read_table("nyc_taxi_zones", "Silver_lakehouse")
    unknown_location_id, outside_nyc_location_id = get_special_location_ids(zones_df)
    valid_location_ids = get_valid_location_ids(zones_df)
    sedona = prepare_spatial_processing(spark, TAXI_ZONES_SHAPEFILE_FOLDER_PATH)
    for file_path in new_files[-24:]:#new_files[:1]
        df = load_and_process_taxi_table(file_path, YELLOW_TRIPDATA_COLUMN_MAPPING, YELLOW_TRIPDATA_SCHEMA)
        df = add_zone_from_coordinates_or_location(df, sedona)
        df = finalize_taxi_locations(df, YELLOW_TRIPDATA_RESULT_COLUMNS, unknown_location_id, outside_nyc_location_id, valid_location_ids)
        save_table(df, table_name, "Silver_lakehouse", "append")

StatementMeta(, 46969f7f-d639-49aa-a349-640fb5c69e39, 77, Finished, Available, Finished, False)

In [52]:
process(YELLOW_TRIPDATA_FOLDER_PATH)

StatementMeta(, 46969f7f-d639-49aa-a349-640fb5c69e39, 75, Finished, Available, Finished, False)